In [1]:
import numpy as np
import pandas as pd
import torch
import pickle
from pathlib import Path

import MF_class as MF

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

# 1. Choose Dataset

In [2]:
datasets = ['ml-1m', 'steam', 'goodreads', 'ml-10m']
DATASET = datasets[3]

In [3]:
parameters_dict = {
    'ml-1m': {
        'n_factors': 40,
        'lr': 1e-3,
        'batch_size': 2**15,
        'n_epochs': 25},
    'steam': {
        'n_factors': 50,
        'lr': 2e-3,
        'batch_size': 2**16,
        'n_epochs': 30},
    'goodreads': {
        'n_factors': 50,
        'lr': 1e-3,
        'batch_size': 2**15,
        'n_epochs': 20},
    'ml-10m': {
        'n_factors': 40,
        'lr': 5e-3,
        'batch_size': 2**15,
        'n_epochs': 30},
}

n_factors  = parameters_dict[DATASET]['n_factors']
lr         = parameters_dict[DATASET]['lr']
batch_size = parameters_dict[DATASET]['batch_size']
n_epochs   = parameters_dict[DATASET]['n_epochs']

# 2. Load Data

In [ ]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET

data = pd.read_csv(data_path / 'data_clean.csv')

n_users = data['user_id'].nunique()
n_items = data['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

Number of users: 5369, Number of items: 3362


In [5]:
data['interaction'] = 1
data_matrix = data.pivot_table(index='user_id', columns='item_id', values='interaction').fillna(0).astype(int)
melted = data_matrix.melt(ignore_index=False).reset_index()

In [6]:
rng = np.random.default_rng(seed=42)
valid_idx = rng.choice(melted.index, size=int(0.1 * len(melted)), replace=False)
train_idx = melted.index.difference(valid_idx)

train = melted.loc[train_idx].reset_index(drop=True)
valid = melted.loc[valid_idx].reset_index(drop=True)

# 3. Train Model

In [7]:
model = MF.MatrixFactorizationTorch(
    n_users=n_users, 
    n_items=n_items, 
    n_factors=n_factors
)

model.fit(
    train_data=train.values,
    val_data=valid.values,
    lr=lr, 
    wd=1e-7,
    pos_weight=1,
    batch_size=batch_size,
    n_epochs=n_epochs,
    device=torch.device('cuda:0'), 
    use_amp=True)

Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || BCE    | BCE-POS | BCE-NEG | MPR    || BCE    | BCE-POS | BCE-NEG | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
   1   || 0.3047 |  0.9655 |  0.1517 | 0.8168 || 0.3095 |  0.9839 |  0.1536 | 0.8130 || 197.06  | None  | 00:03.06
   2   || 0.2747 |  0.8612 |  0.1390 | 0.8356 || 0.2836 |  0.8939 |  0.1425 | 0.8299 ||  93.03  | 0.420 | 00:05.85
   3   || 0.2636 |  0.8209 |  0.1346 | 0.8418 || 0.2758 |  0.8651 |  0.1395 | 0.8346 ||  55.92  | 0.550 | 00:08.65
   4   || 0.2564 |  0.7952 |  0.1317 | 0.8456 || 0.2719 |  0.8508 |  0.1381 | 0.8368 ||  44.64  | 0.579 | 00:11.43
   5   || 0.2517 |  0.7816 |  0.1291 | 0.8480 || 0.2699 |  0.8463 |  0.1366 | 0.8380 ||  36.64  | 0.523 | 00:14.19
   6   || 0.2488 |  0.7688 |  0.1284 | 0.8495 || 0.2686 |  0.8393 |  0.1366 |

### Save Model

In [8]:
name = f'MF_model_{DATASET}'

model_path = base_artifacts / 'MF_Models'
model.save(path=model_path / (name + '.pt'), note=None)

dict_out = {
    'n_users': n_users,
    'n_items': n_items,
    'n_factors': n_factors,
}

with open(model_path / f'MF_params_{DATASET}.pkl', 'wb') as f:
    pickle.dump(dict_out, f)

### Load Model

In [9]:
model_path = base_artifacts / 'MF_Models'
with open(model_path / f'MF_params_{DATASET}.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

In [10]:
name = f'MF_model_{DATASET}'

loaded_model = MF.MatrixFactorizationTorch(
    n_users=loaded_params['n_users'], 
    n_items=loaded_params['n_items'], 
    n_factors=loaded_params['n_factors']
)

model_path = base_artifacts / 'MF_Models'
loaded_model.load(path=model_path / (name + '.pt'))

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            5369
Number of items:            3362
Number of factors:          40
Learning rate:              0.005
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           30
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-08-07 08:40:33
